# Visualization — method comparison results

Generates histograms, heatmaps, and summary tables comparing all simulation
methods against experimental spectra.

**Workflow:**
1. Run the **QCxMS variant comparison** (section below) to pick the best variant.
2. Set `QCXMS_CANONICAL` in the config cell.
3. Run **Exploration** (all methods, all QCxMS variants visible).
4. Run **Presentation** (canonical QCxMS only, renamed to "QCxMS").

In [ ]:
import os, sys
import pandas as pd
import numpy as np
from pathlib import Path

sys.path.insert(0, os.path.abspath(".."))

from src.visualization.plot_results import (
    run_all, plot_cross_dataset_umap, PALETTE_FULL, PALETTE_PRESENTATION
)

DATA_ROOT = os.path.abspath("../data/simulation_results")
PROC_ROOT = os.path.abspath("../data/processed")

# ── Canonical QCxMS variant (update after reviewing variant comparison below) ─
QCXMS_CANONICAL = "QCxMS_10_ps"

# ── Dataset registry ──────────────────────────────────────────────────────────
# Each entry drives run_all (explore / paper / presentation) and the cross-dataset UMAP.
# Add new datasets here; the loop cells pick them up automatically.
DATASETS = [
    {
        "label":         "Franklin TMS",
        "base_dir":      f"{DATA_ROOT}/franklin_tms/",
        "explore_dir":   "../reports/franklin_tms/explore",
        "paper_dir":     "../reports/franklin_tms/paper",
        "pres_dir":      "../reports/franklin_tms/pres",
        "processed_csv": f"{PROC_ROOT}/franklin_tms/franklin_tms_TMS.csv",
        "smiles_col":    "Original_SMILES",
        "marker":        "o",
        # Molecules where automated TMS assignment disagrees with the reference.
        # Scores are substituted in-memory from the underivatised Franklin run.
        "tms_mismatch": {
            "mol_indices":      ["0007", "0026", "0058"],
            "fallback_sim_dir": f"{DATA_ROOT}/franklin/",
        },
    },
    {
        "label":         "Franklin",
        "base_dir":      f"{DATA_ROOT}/franklin/",
        "explore_dir":   "../reports/franklin/explore",
        "paper_dir":     "../reports/franklin/paper",
        "pres_dir":      "../reports/franklin/pres",
        "processed_csv": f"{PROC_ROOT}/franklin/dataset_unique.csv",
        "smiles_col":    "SMILES",
        "marker":        "s",
    },
    {
        "label":         "UCB-GLOBES tracers",
        "base_dir":      f"{DATA_ROOT}/ucb_globes_tracers/",
        "explore_dir":   "../reports/ucb_globes_tracers/explore",
        "paper_dir":     "../reports/ucb_globes_tracers/paper",
        "pres_dir":      "../reports/ucb_globes_tracers/pres",
        "processed_csv": f"{PROC_ROOT}/ucb_globes_tracers/ucb_globes_tracers.csv",
        "smiles_col":    "Original_SMILES",
        "marker":        "^",
    },
]

# TMS-derivatised datasets pooled for cross-dataset UMAP
UMAP_DATASETS = [DATASETS[0], DATASETS[2]]

# Cross-dataset outputs (UMAP etc.) — not tied to a single dataset folder
COMBINED_PAPER_DIR = "../reports/combined/paper"
COMBINED_PRES_DIR  = "../reports/combined/pres"

## Exploration — all variants

In [ ]:
# Exploration — all variants visible, all datasets
explore_data = {}
for ds in DATASETS:
    if not Path(ds["base_dir"]).exists():
        print(f"Skipping {ds['label']} — results not ready.")
        continue
    print(f"\n{'='*60}\n{ds['label']}\n{'='*60}")
    explore_data[ds["label"]] = run_all(
        base_dir        = ds["base_dir"],
        output_dir      = ds["explore_dir"],
        palette         = PALETTE_FULL,
        qcxms_canonical = QCXMS_CANONICAL,
        tms_mismatch    = ds.get("tms_mismatch"),
    )

In [ ]:
# Paper figures — ACS 2-column (7.08"), 9 pt fonts, 300 DPI, all datasets
paper_data = {}
for ds in DATASETS:
    if not Path(ds["base_dir"]).exists():
        print(f"Skipping {ds['label']} — results not ready.")
        continue
    print(f"\n{'='*60}\n{ds['label']}\n{'='*60}")
    paper_data[ds["label"]] = run_all(
        base_dir        = ds["base_dir"],
        output_dir      = ds["paper_dir"],
        paper           = True,
        qcxms_canonical = QCXMS_CANONICAL,
        tms_mismatch    = ds.get("tms_mismatch"),
    )

## Paper & Presentation figures

Paper mode: ACS 2-column width (7.08"), 9 pt fonts, 300 DPI — canonical QCxMS only.  
Presentation mode: slide-optimised (16–18 pt fonts, larger markers) — canonical QCxMS only.

In [ ]:
# Presentation figures — slide-optimised, all datasets
for ds in DATASETS:
    if not Path(ds["base_dir"]).exists():
        print(f"Skipping {ds['label']} — results not ready.")
        continue
    print(f"\n{'='*60}\n{ds['label']}\n{'='*60}")
    run_all(
        base_dir        = ds["base_dir"],
        output_dir      = ds["pres_dir"],
        presentation    = True,
        qcxms_canonical = QCXMS_CANONICAL,
        tms_mismatch    = ds.get("tms_mismatch"),
    )

## Mirror plots — example molecules (Franklin TMS)

Side-by-side mirror spectra (simulated ↑ / experimental ↓). Scores are 1-bin cosines with tms_mismatch correction applied.

| Mol | Compound | QCxMS | NEIMS | Story |
|-----|----------|:---:|:---:|-------|
| 0003 | 3-Aminocarbazole | **971** | 790 | QCxMS good — pure aromatic, no TMS |
| 0048 | Phthalic acid ester | 157 | **891** | QCxMS bad / NEIMS good — phthalate diester, ester fragmentation |
| 0002 | 5,6,7,7a-tetrahydro-4,4,7a-trimethylbenzofuranone | 321 | 470 | Both struggle — cyclic lactone |

Figures saved to `reports/franklin_tms/pres/`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import pandas as pd
from pathlib import Path

# ── Config ────────────────────────────────────────────────────────────────────
_FT_SIM_ROOT  = Path(DATA_ROOT) / "franklin_tms"
_F_SIM_ROOT   = Path(DATA_ROOT) / "franklin"
_FT_PRES_DIR  = Path(DATASETS[0]["pres_dir"])
_FT_PRES_DIR.mkdir(parents=True, exist_ok=True)

TMS_MISMATCH_MOLS = {"0007", "0026", "0058"}

MIRROR_EXAMPLES = [
    {
        "mol":     "0003",
        "name":    "3-Aminocarbazole",
        "note":    "no TMS · pure aromatic",
        "methods": [("QCxMS_10_ps", "QCxMS"), ("NEIMS", "NEIMS")],
    },
    {
        "mol":     "0048",
        "name":    "Phthalic acid ester",
        "note":    "phthalate diester · ester fragmentation",
        "methods": [("QCxMS_10_ps", "QCxMS"), ("NEIMS", "NEIMS")],
    },
    {
        "mol":     "0002",
        "name":    "Dihydrobenzofuranone",
        "note":    "cyclic lactone · both methods struggle",
        "methods": [("QCxMS_10_ps", "QCxMS"), ("NEIMS", "NEIMS")],
    },
]

METHOD_COLORS = {"QCxMS": "#3B82F6", "NEIMS": "#F59E0B"}
REF_COLOR     = "#888888"
GREY_TEXT_M   = "#4C4C4C"

# ── Print TMS SMILES for each example ────────────────────────────────────────
_tms_csv = pd.read_csv(f"{PROC_ROOT}/franklin_tms/franklin_tms_TMS.csv")
_orig_csv = pd.read_csv(DATASETS[0]["processed_csv"])

print("=" * 60)
print("TMS SMILES for mirror plot examples")
print("=" * 60)
for ex in MIRROR_EXAMPLES:
    mol_idx = int(ex["mol"])
    orig_smi = _orig_csv.loc[mol_idx, "Original_SMILES"] if "Original_SMILES" in _orig_csv.columns else "N/A"
    tms_smi  = _tms_csv.loc[mol_idx, "Modified_SMILES"] if "Modified_SMILES" in _tms_csv.columns and mol_idx < len(_tms_csv) else "N/A"
    print(f"\n{ex['mol']} — {ex['name']}")
    print(f"  Original SMILES : {orig_smi}")
    print(f"  TMS SMILES      : {tms_smi}")
print("=" * 60)


def _load_spec(csv_path):
    df = pd.read_csv(csv_path, header=None, names=["mz", "intensity"])
    mz   = df["mz"].values.astype(float)
    ints = df["intensity"].values.astype(float)
    ints = ints / ints.max() * 999
    return mz, ints


def _get_cosine_1bin(sim_root, mol, method_dir, fallback_root=None):
    root = fallback_root if (fallback_root and mol in TMS_MISMATCH_MOLS) else sim_root
    score_csv = root / "results" / mol / "spectra_all_comparison.csv"
    if not score_csv.exists():
        return float("nan")
    df = pd.read_csv(score_csv)
    row = df[df["Method"] == method_dir]
    return float(row["Cosine"].iloc[0]) if not row.empty else float("nan")


def plot_mirror_pair(ex, sim_root, f_root, pres_dir):
    mol = ex["mol"]
    use_fallback = mol in TMS_MISMATCH_MOLS

    n = len(ex["methods"])
    fig, axes = plt.subplots(1, n, figsize=(5.5 * n, 4.8), facecolor="none")
    if n == 1:
        axes = [axes]

    for ax, (method_dir, method_label) in zip(axes, ex["methods"]):
        ax.set_facecolor("none")

        root = f_root if (method_label == "QCxMS" and use_fallback) else sim_root

        exp_path = root / "EXP" / mol / "spectra" / "spectra_all.csv"
        sim_path = root / method_dir / mol / "spectra" / "spectra_all.csv"

        if not exp_path.exists() or not sim_path.exists():
            ax.text(0.5, 0.5, f"No data\n{method_label}", ha="center", va="center",
                    transform=ax.transAxes, color=GREY_TEXT_M)
            ax.set_title(method_label, fontsize=14, color=GREY_TEXT_M)
            continue

        exp_mz, exp_int = _load_spec(exp_path)
        sim_mz, sim_int = _load_spec(sim_path)
        cosine = _get_cosine_1bin(sim_root, mol, method_dir,
                                   fallback_root=f_root if method_label == "QCxMS" else None)
        color  = METHOD_COLORS.get(method_label, "#4472C4")

        ax.vlines(sim_mz,  0,  sim_int, color=color,     linewidth=1.2)
        ax.vlines(exp_mz,  0, -exp_int, color=REF_COLOR, linewidth=1.0)
        ax.axhline(0, color="black", linewidth=0.8)

        ax.yaxis.set_major_formatter(mticker.FuncFormatter(
            lambda v, _: str(int(abs(v)))
        ))

        cosine_str = f"{cosine:.0f}" if not np.isnan(cosine) else "N/A"
        ax.set_title(f"{method_label}  (Cosine = {cosine_str})",
                     fontsize=14, color=GREY_TEXT_M, pad=6)
        ax.set_xlabel("m/z", fontsize=12, color=GREY_TEXT_M)
        ax.set_ylabel("Relative intensity", fontsize=12, color=GREY_TEXT_M)
        ax.tick_params(colors=GREY_TEXT_M, labelsize=11)
        for spine in ax.spines.values():
            spine.set_edgecolor("#cccccc")
        ax.set_xlim(left=0)

        h_sim = mpatches.Patch(color=color,     label=method_label)
        h_ref = mpatches.Patch(color=REF_COLOR, label="Reference")
        leg = ax.legend(handles=[h_sim, h_ref],
                        fontsize=11, frameon=False,
                        loc="upper center",
                        bbox_to_anchor=(0.5, -0.14),
                        ncol=2, handlelength=1.2)
        for t in leg.get_texts():
            t.set_color(GREY_TEXT_M)

    fig.suptitle(f"{mol} — {ex['name']}  ({ex['note']})",
                 fontsize=14, color=GREY_TEXT_M, y=1.02)
    plt.tight_layout()

    out = pres_dir / f"mirror_{mol}.png"
    fig.savefig(out, dpi=300, bbox_inches="tight")
    fig.savefig(pres_dir / f"mirror_{mol}.svg", bbox_inches="tight")
    print(f"Saved: {out}")
    plt.show()
    plt.close()


for ex in MIRROR_EXAMPLES:
    plot_mirror_pair(ex, _FT_SIM_ROOT, _F_SIM_ROOT, _FT_PRES_DIR)

In [ ]:
# Franklin and Franklin TMS are now included in the loops above.
# This cell is kept as a placeholder — re-run the loops above to regenerate all figures.

## QCxMS variant comparison

Compare QCxMS_10_ps, QCxMS_25_ps, and QCxMS_10_ps_iee03 against experimental
spectra to select the best-performing variant. Set `QCXMS_CANONICAL` in the
next cell based on these results.

In [ ]:
QCXMS_VARIANTS = ["QCxMS_10_ps", "QCxMS_25_ps", "QCxMS_10_ps_iee03"]
METRICS        = ["Cosine", "Weighted_Dot", "Tanimoto"]
PEAK_TYPE      = "spectra_all"   # change to spectra_top20 / spectra_10pct as needed

# Local alias — avoids shadowing the global DATASETS registry
COMPARE_DATASETS = {
    "Franklin TMS": f"{DATA_ROOT}/franklin_tms",
    "Franklin":     f"{DATA_ROOT}/franklin",
}

rows = []
for dataset_label, base_dir in COMPARE_DATASETS.items():
    results_dir = Path(base_dir) / "results"
    for variant in QCXMS_VARIANTS:
        scores = {m: [] for m in METRICS}
        for mol_dir in sorted(results_dir.iterdir()):
            if not mol_dir.name.isdigit():
                continue
            csv = mol_dir / f"{PEAK_TYPE}_comparison.csv"
            if not csv.exists():
                continue
            df = pd.read_csv(csv)
            row = df[df["Method"] == variant]
            if row.empty:
                continue
            for m in METRICS:
                scores[m].append(row.iloc[0][m])
        n = len(scores["Cosine"])
        if n == 0:
            continue
        entry = {"Dataset": dataset_label, "Variant": variant, "N": n}
        for m in METRICS:
            vals = pd.Series(scores[m], dtype=float).dropna()
            entry[f"{m} mean"] = vals.mean() if len(vals) > 0 else float("nan")
            entry[f"{m} std"]  = vals.std()  if len(vals) > 0 else float("nan")
        rows.append(entry)

summary = pd.DataFrame(rows)

display_rows = []
for _, r in summary.iterrows():
    display_rows.append({
        "Dataset": r["Dataset"],
        "Variant": r["Variant"],
        "N":       int(r["N"]),
        "Cosine":       f"{r['Cosine mean']:.1f} ± {r['Cosine std']:.1f}",
        "Weighted Dot": f"{r['Weighted_Dot mean']:.1f} ± {r['Weighted_Dot std']:.1f}",
        "Tanimoto":     f"{r['Tanimoto mean']:.3f} ± {r['Tanimoto std']:.3f}",
    })

table = pd.DataFrame(display_rows).set_index(["Dataset", "Variant"])
print(f"Peak type: {PEAK_TYPE}\n")
display(table)

# ── Save LaTeX table to each dataset's paper dir ──────────────────────────────
for ds in [DATASETS[0], DATASETS[1]]:
    out_dir = Path(ds["paper_dir"])
    out_dir.mkdir(parents=True, exist_ok=True)
    ds_rows = [r for r in display_rows if r["Dataset"] == ds["label"]]
    if not ds_rows:
        continue
    tex_rows = []
    for r in ds_rows:
        variant_tex = r["Variant"].replace("_", r"\_")
        tex_rows.append(
            f"  {variant_tex} & {r['N']} & {r['Cosine']} & {r['Weighted Dot']} & {r['Tanimoto']} \\\\"
        )
    tex = "\n".join([
        "% Requires: \\usepackage{booktabs}",
        "\\begin{table}[ht]",
        "\\centering",
        f"\\caption{{Comparison of QCxMS simulation length variants on the {ds['label']} dataset "
        f"(peak type: {PEAK_TYPE}). Mean $\\pm$ std over all molecules with results.}}",
        f"\\label{{tab:qcxms_variants_{ds['label'].lower().replace(' ', '_')}}}",
        "\\begin{tabular}{lcccc}",
        "\\toprule",
        "Variant & $N$ & Cosine & Weighted Dot & Tanimoto \\\\",
        "\\midrule",
        *tex_rows,
        "\\bottomrule",
        "\\end{tabular}",
        "\\end{table}",
    ])
    tex_path = out_dir / "table_qcxms_variants.tex"
    tex_path.write_text(tex)
    print(f"Saved: {tex_path}")

    # Also save to pres_dir without Weighted Dot
    pres_rows = []
    for r in ds_rows:
        variant_tex = r["Variant"].replace("_", r"\_")
        pres_rows.append(
            f"  {variant_tex} & {r['N']} & {r['Cosine']} & {r['Tanimoto']} \\"
        )
    tex_pres = "\n".join([
        "% Requires: \\usepackage{booktabs}",
        "\\begin{table}[ht]",
        "\\centering",
        f"\\caption{{Comparison of QCxMS simulation length variants on the {ds['label']} dataset "
        f"(peak type: {PEAK_TYPE}). Mean $\\pm$ std over all molecules with results.}}",
        f"\\label{{tab:qcxms_variants_{ds['label'].lower().replace(' ', '_')}}}",
        "\\begin{tabular}{lccc}",
        "\\toprule",
        "Variant & $N$ & Cosine & Tanimoto \\\\",
        "\\midrule",
        *pres_rows,
        "\\bottomrule",
        "\\end{tabular}",
        "\\end{table}",
    ])
    pres_out = Path(ds["pres_dir"])
    pres_out.mkdir(parents=True, exist_ok=True)
    pres_tex_path = pres_out / "table_qcxms_variants.tex"
    pres_tex_path.write_text(tex_pres)
    print(f"Saved: {pres_tex_path}")

In [ ]:
# ── Set this after reviewing the table above ─────────────────────────────────
# Options: "QCxMS_10_ps" | "QCxMS_25_ps" | "QCxMS_10_ps_iee03"
QCXMS_CANONICAL = "QCxMS_10_ps"

## ieeatm hyperparameter scan — molecule 0039 (Franklin TMS)

Line plots of similarity metrics vs `ieeatm` (0.1–0.8) for all three peak types,
plus a mirror plot comparing EXP against the best-scoring variant.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from pathlib import Path

SCAN_DIR    = Path(DATA_ROOT) / "franklin_tms" / "0039_10_ps_iee_05"
RESULTS_CSV = SCAN_DIR / "results" / "0039"
EXP_SPECTRA = SCAN_DIR / "EXP" / "0039" / "spectra" / "spectra_all.csv"
IEE_VALUES  = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
IEE_LABELS  = [f"0039_iee{i+1:02d}" for i in range(8)]
SCAN_OUTDIR = Path("../reports/iee_scan")
SCAN_OUTDIR.mkdir(parents=True, exist_ok=True)

GREY_TEXT = "#4C4C4C"
SCAN_METRICS = {
    "Cosine":        "Cosine",
    "Weighted_Dot":  "Weighted Dot",
    "Tanimoto":      "Tanimoto",
    "%S_sim_in_ref": "% sim in ref",
    "%R_ref_in_sim": "% ref in sim",
}

# Colors from make_TMS_derivative_251125_v1.py palette
PT_COLORS = {"all": "#8BD2F8", "top20": "#9DC858", "10pct": "#E05AB1"}
PEAK_LABELS_SCAN = {"all": "All peaks", "top20": "Top 20", "10pct": "≥10%"}

# ── Load comparison data ─────────────────────────────────────────────────────
dfs = {}
for pt in ["all", "top20", "10pct"]:
    csv = RESULTS_CSV / f"spectra_{pt}_comparison.csv"
    if csv.exists():
        df = pd.read_csv(csv).set_index("Method")
        dfs[pt] = df.reindex(IEE_LABELS)

# ── Line plots ────────────────────────────────────────────────────────────────
# 2-row × 3-col layout; last panel hidden
NCOLS, NROWS = 3, 2
fig, axes = plt.subplots(NROWS, NCOLS, figsize=(7.0, 3.6),
                         constrained_layout=True, facecolor="none")
metrics_list = list(SCAN_METRICS.items())
legend_handles, legend_labels = [], []

# Sparse x-ticks: every other value to avoid overlap
TICK_VALS = IEE_VALUES[::2]  # 0.1, 0.3, 0.5, 0.7

for idx, (metric, label) in enumerate(metrics_list):
    row, col = divmod(idx, NCOLS)
    ax = axes[row, col]
    ax.set_facecolor("none")
    for pt in ["all", "top20", "10pct"]:
        if pt not in dfs or metric not in dfs[pt].columns:
            continue
        vals = dfs[pt][metric].values.astype(float)
        color = PT_COLORS[pt]
        h, = ax.plot(IEE_VALUES, vals, "-", color=color, linewidth=1.5,
                     marker="o", markersize=4, label=PEAK_LABELS_SCAN[pt])
        if idx == 0:
            legend_handles.append(h)
            legend_labels.append(PEAK_LABELS_SCAN[pt])
    ax.set_xlabel("eV/atom", fontsize=7, color=GREY_TEXT)
    ax.set_ylabel(label, fontsize=7, color=GREY_TEXT)
    ax.set_xticks(TICK_VALS)
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f"))
    ax.tick_params(colors=GREY_TEXT, labelsize=7)
    ax.yaxis.grid(True, alpha=0.3, linewidth=0.6)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_edgecolor("#cccccc")

# Hide unused panel
axes[1, 2].set_visible(False)

fig.legend(legend_handles, legend_labels,
           fontsize=7, frameon=False, loc="upper center",
           bbox_to_anchor=(0.5, 1.04), ncol=3)

out = SCAN_OUTDIR / "iee_scan_metrics.svg"
plt.savefig(out, format="svg", bbox_inches="tight")
plt.savefig(SCAN_OUTDIR / "iee_scan_metrics.png", dpi=300, bbox_inches="tight")
print(f"Saved: {out}")
plt.show()
plt.close()

# ── Mirror plot — EXP vs best iee variant (by Cosine, spectra_all) ───────────
if "all" in dfs and "Cosine" in dfs["all"].columns:
    best_label = dfs["all"]["Cosine"].idxmax()
    best_iee   = IEE_VALUES[IEE_LABELS.index(best_label)]
    best_idx   = IEE_LABELS.index(best_label)
    sim_spectra_csv = (
        SCAN_DIR / best_label / "GS-opt" / "MS-run" / "spectra" / "spectra_top20.csv"
    )
    exp_df = pd.read_csv(EXP_SPECTRA, header=None, names=["mz", "intensity"])
    sim_df = pd.read_csv(sim_spectra_csv, header=None, names=["mz", "intensity"])

    fig, ax = plt.subplots(figsize=(3.35, 2.8), facecolor="none")
    ax.set_facecolor("none")
    ax.bar(exp_df["mz"],  exp_df["intensity"],  width=1.0,
           color="#5A99D3", alpha=0.9, label="Experimental")
    ax.bar(sim_df["mz"], -sim_df["intensity"], width=1.0,
           color="#D36EA5", alpha=0.9, label=f"QCxMS ieeatm={best_iee:.1f}")
    ax.axhline(0, color="#cccccc", linewidth=0.8)
    ax.set_xlabel("m/z", fontsize=8, color=GREY_TEXT)
    ax.set_ylabel("Intensity (base peak = 999)", fontsize=8, color=GREY_TEXT)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{abs(int(v))}"))
    ax.tick_params(colors=GREY_TEXT, labelsize=8)
    for spine in ax.spines.values():
        spine.set_edgecolor("#cccccc")
    leg = ax.legend(fontsize=7, frameon=False)
    for t in leg.get_texts():
        t.set_color(GREY_TEXT)
    plt.tight_layout()
    out_m = SCAN_OUTDIR / f"mirror_0039_iee{best_idx+1:02d}.svg"
    plt.savefig(out_m, format="svg", bbox_inches="tight")
    plt.savefig(SCAN_OUTDIR / f"mirror_0039_iee{best_idx+1:02d}.png", dpi=300, bbox_inches="tight")
    print(f"Saved: {out_m}")
    plt.show()
    plt.close()

## Context comparison plot (DiffEIMS vs DiffMS)

Stand-alone bar chart comparing external benchmark methods — edit values as needed.

## Strict-overlap comparison

Results restricted to molecules where **all active methods and EXP** have complete spectra.
Reported alongside the full per-method results as a sensitivity check.

In [ ]:
from src.visualization.plot_results import load_results, print_summary_tables, get_active_methods, METRIC_LABELS, METRICS

# ── Peak-type display labels and order ───────────────────────────────────────
PT_LABELS = {"all": "All", "top20": "Top 20", "10pct": r"$\geq$10\%"}
PT_ORDER  = ["all", "top20", "10pct"]


def strict_overlap_tables(base_dir, output_dir, qcxms_canonical=QCXMS_CANONICAL,
                           palette=None, presentation=False, tms_mismatch=None):
    """Print and save tables restricted to molecules complete in all methods.

    Saves to output_dir:
      - table_strict_overlap_{pt}.tex  for each peak type
      - table_combined.tex             booktabs combined table (all methods × strategies)
      - sensitivity_figure.png / .svg
    """
    from IPython.display import display
    from pathlib import Path
    import numpy as np
    import pandas as pd

    if palette is None:
        palette = PALETTE_PRESENTATION

    data = load_results(base_dir, presentation=True, qcxms_canonical=qcxms_canonical, tms_mismatch=tms_mismatch)
    method_order = get_active_methods(data, palette)

    # ── Strict-overlap molecule set ───────────────────────────────────────────
    pivot = data.groupby(["Molecule", "Method"])["Cosine"].mean().unstack("Method")
    complete_mols = pivot.dropna(subset=method_order).index.tolist()
    print(f"Strict-overlap molecules ({len(complete_mols)}/{data['Molecule'].nunique()} total):")
    print(complete_mols)

    subset = data[data["Molecule"].isin(complete_mols)]
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    # cell_data[method][pt][metric] = (mean, std)
    cell_data = {m: {pt: {} for pt in PT_ORDER} for m in method_order}

    for pt in PT_ORDER:
        # Exclude always-NaN metrics (e.g. Entropy_Similarity) from dropna
        # so that empty columns do not eliminate all rows from the table.
        _nonempty = [m for m in METRICS if subset[m].notna().any()]
        pt_data = subset[subset["Peak_Type"] == pt].dropna(subset=_nonempty)
        rows, latex_rows = [], []
        for method in method_order:
            md = pt_data[pt_data["Method"] == method]
            row      = {"Method": method}
            latex_row = {"Method": method.replace("_", r"\_")}
            for metric in METRICS:
                vals = md[metric].dropna()
                if len(vals) > 0:
                    mean, std = vals.mean(), vals.std()
                    row[metric]      = f"{mean:.1f} ± {std:.1f}"
                    latex_row[metric] = f"{mean:.1f} $\\pm$ {std:.1f}"
                    cell_data[method][pt][metric] = (mean, std)
                else:
                    row[metric] = latex_row[metric] = "N/A"
                    cell_data[method][pt][metric] = (float("nan"), float("nan"))
            rows.append(row)
            latex_rows.append(latex_row)

        df_table = pd.DataFrame(rows).set_index("Method")
        df_table.columns = [METRIC_LABELS[m] for m in df_table.columns]
        print(f"\n=== Strict overlap — Peak Type: {pt} (N={len(complete_mols)}) ===")
        display(df_table)

        df_latex = pd.DataFrame(latex_rows).set_index("Method")
        df_latex.columns = [METRIC_LABELS[m] for m in df_latex.columns]
        tex_path = Path(output_dir) / f"table_strict_overlap_{pt}.tex"
        tex_path.write_text(df_latex.to_latex(escape=False))
        print(f"  Saved: {tex_path}")

    # ── Combined booktabs table ───────────────────────────────────────────────
    pres_metrics  = [m for m in METRICS if m != "Weighted_Dot"]
    table_metrics = pres_metrics if presentation else METRICS
    metric_labels = [METRIC_LABELS[m] for m in table_metrics]
    col_fmt = "ll" + "c" * len(table_metrics)
    header  = " & ".join(["Method", "Strategy"] + metric_labels)

    lines = []
    lines.append("% Requires: \\usepackage{booktabs}, \\usepackage{multirow}")
    lines.append("\\begin{table}[ht]")
    lines.append("\\centering")
    lines.append("\\caption{Benchmark results: mean similarity scores (\\%) for each method and peak-picking strategy on the strict-overlap molecule set ($N=" + str(len(complete_mols)) + "$). Methods are grouped by type: physics-based (QCxMS, QCxMS2) and ML-based (CFMID, NEIMS).}\\label{tab:benchmark_results}")
    lines.append(f"\\begin{{tabular}}{{{col_fmt}}}")
    lines.append("\\toprule")
    lines.append(header + " \\\\")
    lines.append("\\midrule")

    for m_idx, method in enumerate(method_order):
        if m_idx > 0:
            lines.append("\\midrule")
        n_pts = len(PT_ORDER)
        for p_idx, pt in enumerate(PT_ORDER):
            method_cell = (f"\\multirow{{{n_pts}}}{{*}}{{{method.replace('_', ' ')}}}"
                           if p_idx == 0 else "")
            metric_cells = []
            for metric in table_metrics:
                mean, std = cell_data[method][pt].get(metric, (float("nan"), float("nan")))
                metric_cells.append("N/A" if np.isnan(mean) else f"{mean:.1f} $\\pm$ {std:.1f}")
            lines.append(" & ".join([method_cell, PT_LABELS[pt]] + metric_cells) + " \\\\")

    lines.append("\\bottomrule")
    lines.append("\\end{tabular}")
    lines.append("\\end{table}")

    combined_tex = "\n".join(lines)
    combined_path = Path(output_dir) / "table_combined.tex"
    combined_path.write_text(combined_tex)
    print(f"\n  Saved combined table: {combined_path}")

    # ── Sensitivity figure ────────────────────────────────────────────────────
    import matplotlib.pyplot as plt

    GREY_TEXT_S = "#4C4C4C"
    x_pos    = list(range(len(PT_ORDER)))
    x_labels = ["All", "Top 20", r"$\geq$10\%"]

    if presentation:
        fig_w, fig_h   = 13.0, 4.5
        tick_fs, lbl_fs, leg_fs = 13, 13, 12
        lw, ms, caps   = 2.0, 7, 3
        leg_anchor     = (0.5, -0.18)
    else:
        fig_w, fig_h   = 7.08, 2.5
        tick_fs, lbl_fs, leg_fs = 7, 7, 7
        lw, ms, caps   = 1.2, 4, 2
        leg_anchor     = (0.5, -0.22)

    fig, axes = plt.subplots(1, len(METRICS), figsize=(fig_w, fig_h),
                             sharey=False, facecolor="none")
    for ax, metric in zip(axes, METRICS):
        label = METRIC_LABELS[metric]
        ax.set_facecolor("none")
        for method in method_order:
            color  = palette.get(method, "#888888")
            y_vals = [cell_data[method][pt][metric][0] for pt in PT_ORDER]
            y_errs = [cell_data[method][pt][metric][1] for pt in PT_ORDER]
            ax.errorbar(x_pos, y_vals, yerr=y_errs,
                        color=color, linewidth=lw, marker="o", markersize=ms,
                        capsize=caps, elinewidth=lw*0.7, label=method)
        ax.set_xticks(x_pos)
        ax.set_xticklabels(x_labels, rotation=30, ha="right", fontsize=tick_fs)
        ax.set_ylabel(label, fontsize=lbl_fs, color=GREY_TEXT_S)
        ax.tick_params(colors=GREY_TEXT_S, labelsize=tick_fs)
        for spine in ax.spines.values():
            spine.set_edgecolor("#cccccc")

    handles, labels_leg = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels_leg, fontsize=leg_fs, frameon=False,
               loc="lower center", bbox_to_anchor=leg_anchor,
               ncol=len(method_order))
    plt.tight_layout()
    sens_png = Path(output_dir) / "sensitivity_figure.png"
    sens_svg = Path(output_dir) / "sensitivity_figure.svg"
    plt.savefig(sens_png, dpi=300, bbox_inches="tight")
    plt.savefig(sens_svg, bbox_inches="tight")
    print(f"  Saved sensitivity figure: {sens_png}")
    plt.show()
    plt.close()




def _pres_combined_table(base_dir, output_dir, qcxms_canonical=QCXMS_CANONICAL, palette=None, tms_mismatch=None):
    """Combined table for presentation: full per-method dataset, no Weighted Dot, $\%$ in headers."""
    from pathlib import Path
    import numpy as np
    import pandas as pd

    if palette is None:
        palette = PALETTE_PRESENTATION

    data     = load_results(base_dir, presentation=True, qcxms_canonical=qcxms_canonical, tms_mismatch=tms_mismatch)
    methods  = get_active_methods(data, palette)
    p_metrics = [m for m in METRICS if m != "Weighted_Dot"]

    # LaTeX-safe labels with $\%$ for percent columns
    def _tex_label(m):
        lbl = METRIC_LABELS[m]
        return lbl.replace("%", "$\%$")

    col_fmt = "ll" + "c" * len(p_metrics)
    header  = " & ".join(["Method", "Strategy"] + [_tex_label(m) for m in p_metrics])

    # Compute per-method × peak-type means on full dataset
    cell_data = {}
    for method in methods:
        cell_data[method] = {}
        md = data[data["Method"] == method]
        for pt in PT_ORDER:
            pt_data = md[md["Peak_Type"] == pt]
            cell_data[method][pt] = {}
            for metric in p_metrics:
                vals = pt_data[metric].dropna()
                cell_data[method][pt][metric] = (
                    (vals.mean(), vals.std()) if len(vals) > 0
                    else (float("nan"), float("nan"))
                )

    lines = []
    lines.append("% Requires: \\usepackage{booktabs}, \\usepackage{multirow}")
    lines.append("\\begin{table}[ht]")
    lines.append("\\centering")
    lines.append("\\caption{Benchmark results: mean similarity scores for each method and peak-picking strategy (all molecules with results per method).}\\label{tab:benchmark_results_pres}")
    lines.append(f"\\begin{{tabular}}{{{col_fmt}}}")
    lines.append("\\toprule")
    lines.append(header + " \\\\")
    lines.append("\\midrule")

    for m_idx, method in enumerate(methods):
        if m_idx > 0:
            lines.append("\\midrule")
        for p_idx, pt in enumerate(PT_ORDER):
            method_cell = (
                f"\\multirow{{{len(PT_ORDER)}}}{{*}}{{{method.replace(chr(95), ' ')}}}"
                if p_idx == 0 else ""
            )
            cells = []
            for metric in p_metrics:
                mean, std = cell_data[method][pt][metric]
                cells.append("N/A" if np.isnan(mean) else f"{mean:.1f} $\\pm$ {std:.1f}")
            lines.append(" & ".join([method_cell, PT_LABELS[pt]] + cells) + " \\\\")

    lines.append("\\bottomrule")
    lines.append("\\end{tabular}")
    lines.append("\\end{table}")

    Path(output_dir).mkdir(parents=True, exist_ok=True)
    out = Path(output_dir) / "table_combined.tex"
    out.write_text("\n".join(lines))
    print(f"  Saved pres combined table: {out}")


# ── Run for all datasets → paper_dir (strict overlap) + pres_dir (full data) ─
for ds in DATASETS:
    if not Path(ds["base_dir"]).exists():
        print(f"Skipping {ds['label']} — results not ready.")
        continue
    print(f"\n{'='*60}\n{ds['label']}\n{'='*60}")
    strict_overlap_tables(ds["base_dir"], ds["paper_dir"], tms_mismatch=ds.get("tms_mismatch"))
    strict_overlap_tables(ds["base_dir"], ds["pres_dir"], presentation=True, tms_mismatch=ds.get("tms_mismatch"))
    _pres_combined_table(ds["base_dir"], ds["pres_dir"], tms_mismatch=ds.get("tms_mismatch"))

## Spectra completeness diagnostics

Coverage table showing how many molecules each method has complete spectra for.
Saves `diagnostics_coverage.tex` and `diagnostics_missing.tex` to each output directory.

In [ ]:
from src.analysis.diagnose_spectra import diagnose_spectra

DIAG_DATASETS = [
    "QCxMS_10_ps", "QCxMS_25_ps", "QCxMS_10_ps_iee03",
    "QCxMS2", "NEIMS", "CFMID", "EXP",
]

for ds in [DATASETS[0], DATASETS[1]]:
    if not Path(ds["base_dir"]).exists():
        print(f"Skipping {ds['label']} — results not ready.")
        continue
    print(f"\n{'='*60}\n{ds['label']}\n{'='*60}")
    diagnose_spectra(
        sim_base_dir = ds["base_dir"],
        datasets     = DIAG_DATASETS,
        plot         = False,
        output_dir   = ds["pres_dir"],
    )

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

NAVY      = "#5A99D3"
PINK      = "#D36EA5"
DARK_GREY = "#6B7A8D"
LIGHT_GREY = "#C0C8D4"
GREY_TEXT  = "#4C4C4C"

# ── Data (accuracy % and Tanimoto similarity) ──────────────────────────────
RESULTS = {
    "DIFFEIMS Top-1":  {"accuracy": 6,  "tanimoto": 0.41, "color": NAVY},
    "DIFFEIMS Top-10": {"accuracy": 20, "tanimoto": 0.59, "color": PINK},
    "DiffMS Top-1":    {"accuracy": 8,  "tanimoto": 0.35, "color": DARK_GREY},
    "DiffMS Top-10":   {"accuracy": 15, "tanimoto": 0.47, "color": LIGHT_GREY},
}

x, width, gap = np.arange(2), 0.18, 0.06
offsets = [-1.5*width - gap/2, -0.5*width - gap/2,
            0.5*width + gap/2,  1.5*width + gap/2]

fig, ax = plt.subplots(figsize=(8, 5.5), facecolor="none")
ax.set_facecolor("none")

for (label, vals), offset in zip(RESULTS.items(), offsets):
    data = [vals["accuracy"], vals["tanimoto"] * 100]
    bars = ax.bar(x + offset, data, width, color=vals["color"],
                  label=label, zorder=3, linewidth=0)
    for i, (bar, val) in enumerate(zip(bars, data)):
        txt = f"{int(val)}%" if i == 0 else f"{vals['tanimoto']:.2f}"
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.8,
                txt, ha="center", va="bottom", fontsize=16,
                color=GREY_TEXT, fontweight="bold", fontfamily="Helvetica")

ax.set_xticks(x)
ax.set_xticklabels(["Accuracy", "Tanimoto similarity"],
                   fontsize=16, color=GREY_TEXT, fontfamily="Helvetica")
ax.set_ylim(0, 75)
ax.tick_params(colors=GREY_TEXT, length=0, labelsize=18)
ax.yaxis.set_visible(False)
for spine in ax.spines.values():
    spine.set_visible(False)

leg = ax.legend(frameon=False, fontsize=16, ncol=2,
                loc="upper center", bbox_to_anchor=(0.5, 1.13),
                handlelength=1.4, handleheight=0.9,
                handletextpad=0.5, columnspacing=1.5)
for text in leg.get_texts():
    text.set_color(GREY_TEXT)
    text.set_fontfamily("Helvetica")

plt.tight_layout()
plt.savefig("results_plot.svg", format="svg", bbox_inches="tight", dpi=500)
plt.show()

## Dataset characterisation — derivatization and functional groups

Stacked bar charts showing (1) which functional groups were TMS-derivatized and in what numbers,
and (2) the prevalence of SIMPOL functional groups across the underivatized benchmark molecules.
Style matches `make_TMS_derivative_251125_v1.py`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# ── Style (matches make_TMS_derivative_251125_v1.py) ─────────────────────────
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "font.size": 24, "axes.labelsize": 24,
    "xtick.labelsize": 24, "ytick.labelsize": 24, "legend.fontsize": 24,
})

COLORS = {
    "OH": "#9DC858", "SH": "#3D6B3F", "Secondary Amine": "#E05AB1",
    "Primary Amine": "#F2A85A", "Imine": "#6C5B7B", "OOH": "#E05AB1",
    "COOH": "#8BD2F8", "None": "#7f7f7f",
}

PROC_ROOT_TMS = Path(f"{PROC_ROOT}/franklin_tms")
DERIV_OUT = Path(DATASETS[0]["pres_dir"])
DERIV_OUT.mkdir(parents=True, exist_ok=True)

# ── Load derivatization data ──────────────────────────────────────────────────
df = pd.read_csv(PROC_ROOT_TMS / "franklin_tms_TMS.csv")
df = df.rename(columns={"Primary_Amine": "Primary Amine", "Secondary_Amine": "Secondary Amine"})
df["None"] = (df["Total_Replacements"] == 0).astype(int)

GROUP_COLS = ["OH", "SH", "Secondary Amine", "Primary Amine", "Imine", "OOH", "COOH", "None"]
n_mols = len(df)

# ── Build plot data (replicates plot_substitutions logic) ─────────────────────
plot_data = []
for _, row in df.iterrows():
    total = row["Total_Replacements"]
    if total == 0:
        plot_data.append({"Total_Replacements": 0, "Functional Group": "None",
                          "Molecule Contribution": 1.0 / n_mols})
    else:
        total_groups = sum(row[g] for g in GROUP_COLS if g != "None")
        for g in GROUP_COLS:
            if g != "None" and row[g] > 0:
                plot_data.append({"Total_Replacements": total, "Functional Group": g,
                                  "Molecule Contribution": row[g] / total_groups / n_mols})

plot_df = pd.DataFrame(plot_data)
grouped = (plot_df.groupby(["Total_Replacements", "Functional Group"])["Molecule Contribution"]
           .sum().unstack(fill_value=0))

present = [g for g in GROUP_COLS if g in grouped.columns]

fig, ax = plt.subplots(figsize=(20, 15))
grouped[present].plot(kind="bar", stacked=True, ax=ax,
                      color=[COLORS[g] for g in present],
                      edgecolor="black", linewidth=1)
ax.set_xlabel("Number of TMS substitutions per molecule", labelpad=15)
ax.set_ylabel("Fraction of molecules", labelpad=15)
plt.xticks(rotation=0)
ax.legend(title="", loc="upper right", frameon=True, facecolor="white", edgecolor="black")
plt.tight_layout()
plt.savefig(DERIV_OUT / "tms_derivatization_by_fg.png", dpi=300, bbox_inches="tight")
plt.savefig(DERIV_OUT / "tms_derivatization_by_fg.pdf", bbox_inches="tight")
print(f"Saved: {DERIV_OUT / 'tms_derivatization_by_fg.png'}")
plt.show()
plt.close()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# ── Style (same as derivatization plot) ──────────────────────────────────────
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "font.size": 24, "axes.labelsize": 24,
    "xtick.labelsize": 24, "ytick.labelsize": 24, "legend.fontsize": 24,
})

STACK_COLORS = {"1": "#8BD2F8", "2": "#9DC858", "3+": "#E05AB1"}

_franklin_proc = Path(f"{PROC_ROOT}/franklin")
SIMPOL_OUT = Path(DATASETS[1]["pres_dir"])   # Franklin pres dir
SIMPOL_OUT.mkdir(parents=True, exist_ok=True)

# ── Load SIMPOL data (computed from underivatized SMILES — for dataset characterization) ──
simpol = pd.read_csv(_franklin_proc / "franklin_SIMPOL_benchmark.csv")

NON_FG = {"SMILES", "oxygen_count", "aromatic_ring", "non_aromatic_ring",
           "nitrophenol", "carbon number"}
fg_cols = [c for c in simpol.columns if c not in NON_FG]
n_mols  = len(simpol)

# ── Compute stacked proportions per FG ───────────────────────────────────────
records = []
for fg in fg_cols:
    col = simpol[fg]
    p1  = (col == 1).sum() / n_mols
    p2  = (col == 2).sum() / n_mols
    p3  = (col >= 3).sum() / n_mols
    total = p1 + p2 + p3
    if total < 0.01:
        continue
    records.append({"fg": fg, "1": p1, "2": p2, "3+": p3, "total": total})

df_plot = pd.DataFrame(records).sort_values("total", ascending=True)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(20, max(10, 0.6 * len(df_plot))), facecolor="none")
ax.set_facecolor("none")

lefts = np.zeros(len(df_plot))
y_pos = np.arange(len(df_plot))
for key in ["1", "2", "3+"]:
    vals = df_plot[key].values
    ax.barh(y_pos, vals, left=lefts, height=0.6,
            color=STACK_COLORS[key], edgecolor="black", linewidth=1, label=key)
    lefts += vals

ax.set_yticks(y_pos)
ax.set_yticklabels(df_plot["fg"].str.replace("_", " "), fontsize=24)
ax.set_xlabel("Proportion of benchmark molecules", labelpad=15)
ax.legend(title="Occurrences per molecule", loc="lower right",
          frameon=True, facecolor="white", edgecolor="black")
plt.tight_layout()
plt.savefig(SIMPOL_OUT / "fg_presence_stacked.png", dpi=300, bbox_inches="tight")
plt.savefig(SIMPOL_OUT / "fg_presence_stacked.pdf", bbox_inches="tight")
print(f"Saved: {SIMPOL_OUT / 'fg_presence_stacked.png'}")
plt.show()
plt.close()

## Dataset characterisation — MW, functional groups, TMS distribution

Three-panel figure combining molecular weight distribution, SIMPOL functional group
prevalence (stacked by occurrence count), and TMS substitution count distribution.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import Descriptors

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "font.size": 24, "axes.labelsize": 24,
    "xtick.labelsize": 24, "ytick.labelsize": 24, "legend.fontsize": 20,
})

STACK_COLORS = {"1": "#8BD2F8", "2": "#9DC858", "3+": "#E05AB1"}
OUT_DIR = Path(DATASETS[0]["pres_dir"])
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Load data ─────────────────────────────────────────────────────────────────
df_tms  = pd.read_csv(f"{PROC_ROOT}/franklin_tms/dataset_unique.csv")
simpol  = pd.read_csv(f"{PROC_ROOT}/franklin/franklin_SIMPOL_benchmark.csv")
n_mols  = len(df_tms)

# Molecular weight
df_tms["MW"] = df_tms["SMILES"].apply(
    lambda s: Descriptors.ExactMolWt(Chem.MolFromSmiles(s))
    if s and Chem.MolFromSmiles(s) else None
)

# SIMPOL stacked proportions
NON_FG = {"SMILES", "oxygen_count", "aromatic_ring", "non_aromatic_ring",
          "nitrophenol", "carbon number"}
fg_cols = [c for c in simpol.columns if c not in NON_FG]
records = []
for fg in fg_cols:
    col   = simpol[fg]
    p1, p2, p3 = (col == 1).sum() / n_mols, (col == 2).sum() / n_mols, (col >= 3).sum() / n_mols
    total = p1 + p2 + p3
    if total < 0.01:
        continue
    records.append({"fg": fg, "1": p1, "2": p2, "3+": p3, "total": total})
df_fg = pd.DataFrame(records).sort_values("total", ascending=True)

# TMS distribution — stacked by functional group
deriv_df = pd.read_csv(f"{PROC_ROOT}/franklin_tms/franklin_tms_TMS.csv")
deriv_df = deriv_df.rename(columns={"Primary_Amine": "Primary Amine",
                                     "Secondary_Amine": "Secondary Amine"})
deriv_df["None"] = (deriv_df["Total_Replacements"] == 0).astype(int)

FG_STACK   = ["OH", "COOH", "None"]
FG_COLORS  = {"OH": "#9DC858", "COOH": "#8BD2F8", "None": "#7f7f7f"}

tms_plot_data = []
for _, row in deriv_df.iterrows():
    total = row["Total_Replacements"]
    if total == 0:
        tms_plot_data.append({"Total": 0, "FG": "None", "contrib": 1.0 / n_mols})
    else:
        tot_groups = sum(row[g] for g in ["OH", "COOH"] if g in row.index)
        for g in ["OH", "COOH"]:
            if row.get(g, 0) > 0:
                tms_plot_data.append({"Total": total, "FG": g,
                                       "contrib": row[g] / max(tot_groups, 1) / n_mols})

tms_df = pd.DataFrame(tms_plot_data)
tms_grouped = (tms_df.groupby(["Total", "FG"])["contrib"]
               .sum().unstack(fill_value=0))
tms_present = [g for g in FG_STACK if g in tms_grouped.columns]

# ── 3-panel figure ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(28, max(12, 0.55 * len(df_fg))),
                         gridspec_kw={"width_ratios": [1, 1.8, 0.7]})

# Panel A: Molecular weight distribution
ax = axes[0]
ax.hist(df_tms["MW"].dropna(), bins=20,
        color="#8BD2F8", edgecolor="black", linewidth=1)
ax.set_xlabel("Molecular weight (Da)", labelpad=15)
ax.set_ylabel("Number of molecules", labelpad=15)

# Panel B: FG prevalence (horizontal stacked)
ax = axes[1]
lefts = np.zeros(len(df_fg))
y_pos = np.arange(len(df_fg))
for key in ["1", "2", "3+"]:
    vals = df_fg[key].values
    ax.barh(y_pos, vals, left=lefts, height=0.6,
            color=STACK_COLORS[key], edgecolor="black", linewidth=1, label=key)
    lefts += vals
ax.set_yticks(y_pos)
ax.set_yticklabels(df_fg["fg"].str.replace("_", " "), fontsize=20)
ax.set_xlabel("Proportion of benchmark molecules", labelpad=15)
ax.legend(title="Occurrences per molecule", loc="lower right",
          frameon=True, facecolor="white", edgecolor="black")

# Panel C: TMS distribution stacked by functional group
ax = axes[2]
tms_grouped[tms_present].plot(
    kind="bar", stacked=True, ax=ax,
    color=[FG_COLORS[g] for g in tms_present],
    edgecolor="black", linewidth=1, width=0.6
)
ax.set_xlabel("Number of TMS groups", labelpad=15)
ax.set_ylabel("Fraction of molecules", labelpad=15)
ax.tick_params(axis="x", rotation=0)
ax.legend(title="", loc="upper right", frameon=True,
          facecolor="white", edgecolor="black")

plt.tight_layout()
plt.savefig(OUT_DIR / "dataset_characterization.png", dpi=300, bbox_inches="tight")
plt.savefig(OUT_DIR / "dataset_characterization.pdf", bbox_inches="tight")
print(f"Saved: {OUT_DIR / 'dataset_characterization.png'}")
plt.show()
plt.close()

## Molecular space — cross-dataset UMAP coloured by cosine score

Morgan fingerprints (radius=2, 2048 bits) on parent (underivatised) SMILES, pooled across
Franklin TMS (○) and UCB-GLOBES tracers (△). Fitted on the combined set; each panel shows
one simulation method coloured by cosine score (viridis, 0–1000).

Two output figures:
- **Full (2×2)**: QCxMS, QCxMS2, CFMID, NEIMS — for SI or supplementary
- **Compact (1×2)**: QCxMS + NEIMS — for main text / presentation

In [ ]:
# Full 2×2: all four methods — for SI / supplementary
plot_cross_dataset_umap(
    datasets        = UMAP_DATASETS,
    output_dir      = COMBINED_PAPER_DIR,
    qcxms_canonical = QCXMS_CANONICAL,
    methods         = ["QCxMS", "QCxMS2", "CFMID", "NEIMS"],
    figname         = "umap_all_methods",
    paper           = True,
)

# Compact 1×2: QCxMS + NEIMS — for main text
plot_cross_dataset_umap(
    datasets        = UMAP_DATASETS,
    output_dir      = COMBINED_PAPER_DIR,
    qcxms_canonical = QCXMS_CANONICAL,
    methods         = ["QCxMS", "NEIMS"],
    figname         = "umap_qcxms_neims",
    paper           = True,
)

# Presentation version of compact plot (larger fonts/markers)
plot_cross_dataset_umap(
    datasets        = UMAP_DATASETS,
    output_dir      = COMBINED_PRES_DIR,
    qcxms_canonical = QCXMS_CANONICAL,
    methods         = ["QCxMS", "NEIMS"],
    figname         = "umap_qcxms_neims_pres",
    presentation    = True,
)

## Cross-dataset NEIMS comparison — cosine score boxplot

Boxplot comparing NEIMS cosine score distributions across datasets (all-peaks, 1-bin).
Uses only molecules with complete NEIMS results in each dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

_COMBINED_PRES = Path(COMBINED_PRES_DIR)
_COMBINED_PRES.mkdir(parents=True, exist_ok=True)

GREY_TEXT_B = "#4C4C4C"

# ── Datasets to compare ───────────────────────────────────────────────────────
COMPARE_DS = [
    {
        "label":   "Franklin TMS",
        "base_dir": DATASETS[0]["base_dir"],
        "color":   "#5A99D3",
        "tms_mismatch": DATASETS[0].get("tms_mismatch"),
    },
    {
        "label":   "UCB-GLOBES tracers",
        "base_dir": DATASETS[2]["base_dir"],
        "color":   "#D36EA5",
        "tms_mismatch": None,
    },
]

# ── Load NEIMS cosine scores (all-peaks, 1-bin) ───────────────────────────────
records = []
for ds in COMPARE_DS:
    results_dir = Path(ds["base_dir"]) / "results"
    if not results_dir.exists():
        print(f"Skipping {ds['label']} — results not ready.")
        continue

    tms_mm = ds.get("tms_mismatch")
    fallback_dir = (Path(tms_mm["fallback_sim_dir"]) / "results"
                    if tms_mm else None)
    mismatch_mols = set(tms_mm["mol_indices"]) if tms_mm else set()

    for mol_dir in sorted(results_dir.iterdir()):
        if not mol_dir.name.isdigit():
            continue
        mol_idx = mol_dir.name

        # For tms_mismatch molecules, use the fallback (underivatised) results
        if mol_idx in mismatch_mols and fallback_dir is not None:
            csv = fallback_dir / mol_idx / "spectra_all_comparison.csv"
        else:
            csv = mol_dir / "spectra_all_comparison.csv"

        if not csv.exists():
            continue
        df = pd.read_csv(csv)
        row = df[df["Method"] == "NEIMS"]
        if row.empty:
            continue
        records.append({
            "Dataset": ds["label"],
            "Cosine":  float(row["Cosine"].iloc[0]),
            "color":   ds["color"],
        })

df_box = pd.DataFrame(records)
if df_box.empty:
    print("No NEIMS results found.")
else:
    ds_labels  = [ds["label"] for ds in COMPARE_DS if ds["label"] in df_box["Dataset"].unique()]
    ds_colors  = {ds["label"]: ds["color"] for ds in COMPARE_DS}
    n_per_ds   = df_box.groupby("Dataset")["Cosine"].count()

    fig, ax = plt.subplots(figsize=(max(6.0, 3.5 * len(ds_labels)), 8.0), facecolor="none")
    ax.set_facecolor("none")

    data_by_ds = [df_box[df_box["Dataset"] == lbl]["Cosine"].dropna().values
                  for lbl in ds_labels]

    bp = ax.boxplot(data_by_ds,
                    positions=range(len(ds_labels)),
                    widths=0.45,
                    patch_artist=True,
                    medianprops=dict(color="white", linewidth=2.5),
                    whiskerprops=dict(linewidth=1.5),
                    capprops=dict(linewidth=1.5),
                    flierprops=dict(marker="o", markersize=4, alpha=0.5, linewidth=0))

    for patch, lbl in zip(bp["boxes"], ds_labels):
        patch.set_facecolor(ds_colors[lbl])
        patch.set_alpha(0.85)
        patch.set_linewidth(0)

    for i, (lbl, vals) in enumerate(zip(ds_labels, data_by_ds)):
        ax.text(i, -130, f"median={np.median(vals):.0f}",
                ha="center", va="top", fontsize=11, color=GREY_TEXT_B)

    ax.set_xticks(range(len(ds_labels)))
    ax.set_xticklabels(ds_labels, fontsize=13, color=GREY_TEXT_B, rotation=35, ha="right")
    ax.set_ylabel("Cosine score (NEIMS, all peaks)", fontsize=14, color=GREY_TEXT_B)
    ax.set_ylim(-200, 1050)
    ax.tick_params(colors=GREY_TEXT_B, labelsize=13)
    ax.axhline(0, color="#cccccc", linewidth=0.8)
    for spine in ax.spines.values():
        spine.set_edgecolor("#cccccc")

    plt.tight_layout()
    out = _COMBINED_PRES / "neims_cosine_by_dataset.png"
    plt.savefig(out, dpi=300, bbox_inches='tight')
    plt.savefig(_COMBINED_PRES / "neims_cosine_by_dataset.pdf", bbox_inches='tight')
    print(f"Saved: {out}")
    plt.show()
    plt.close()
